## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [6]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

%matplotlib inline

In [7]:
# РОЗМІР ЕФЕКТУ

retention_aim = 0.2
retention_base = 0.19

effect_size = sms.proportion_effectsize(0.2, 0.19)    # Розрахунок розміру ефекту на основі наших очікуваних показників

print(f"effect_size = {effect_size:.3f}")

effect_size = 0.025


In [8]:
# РОЗМІР ВИБІРКИ
# К-сть користувачів, у кожній групі, залежить від:
# статистичної потужності (зазвичай 0.8),
# рівня значущості ( 𝛼=0.05 ),
# очікуваної величини ефекту.

required_n = sms.NormalIndPower().solve_power(
    effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
    )                                                  # Розрахунок необхідного розміру вибірки

required_n = ceil(required_n)                          # Округлення до наступного цілого числа

print(f"необхідний розмір вибірки = {required_n} користувачів")

необхідний розмір вибірки = 24638 користувачів


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [9]:
df = pd.read_csv("../data/cookie_cats.csv")

df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userid          90189 non-null  int64 
 1   version         90189 non-null  object
 2   sum_gamerounds  90189 non-null  int64 
 3   retention_1     90189 non-null  bool  
 4   retention_7     90189 non-null  bool  
dtypes: bool(2), int64(2), object(1)
memory usage: 2.2+ MB


In [11]:
# СКІЛЬКИ гравців в КОНТРОЛЬНІЙ ГРУПІ і в ЕКСПЕРИМЕНТАЛЬНІЙ

df["version"].value_counts()

version
gate_40    45489
gate_30    44700
Name: count, dtype: int64

In [12]:
# СЕРЕДНЄ ЗНАЧЕННЯ retention_7

df.groupby("version")["retention_7"].mean()

version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64

  0.190201  >  0.182000

/(gate_30 )    (gate_40)/

-> отже стара версія дає краще утримання гравців

ГІПОТЕЗИ

𝐻0: 𝑝 = 𝑝0     різниці немає, gate_30 = gate_40

𝐻𝑎: 𝑝 < 𝑝0     нова версія з воротами на 40 рівні гірша за стару(базову)

де  𝑝  і  𝑝0  позначають коефіцієнт конверсії версій гри

p0​ = контрольна (стара версія, gate_30)

p = тестова (нова версія, gate_40)



рівень значущості 95%

𝛼=0.05






3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [13]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

In [14]:
gate_30_results = df[df["version"] == "gate_30"]["retention_7"]
gate_40_results = df[df["version"] == "gate_40"]["retention_7"]

In [15]:
n_con = gate_30_results.count()
n_treat = gate_40_results.count()
successes = [gate_30_results.sum(), gate_40_results.sum()]
nobs = [n_con, n_treat]

In [16]:
successes


[np.int64(8502), np.int64(8279)]

In [17]:
z_stat, pval = proportions_ztest(successes, nobs=nobs)
(lower_con, lower_treat), (upper_con, upper_treat) = proportion_confint(successes, nobs=nobs, alpha=0.05)

print(f'z statistic: {z_stat:.2f}')
print(f'p-value: {pval:.3f}')
print(f'Довірчий інтервал 95% для групи gate_30: [{lower_con:.3f}, {upper_con:.3f}]')
print(f'Довірчий інтервал 95% для групи gate_40: [{lower_treat:.3f}, {upper_treat:.3f}]')


z statistic: 3.16
p-value: 0.002
Довірчий інтервал 95% для групи gate_30: [0.187, 0.194]
Довірчий інтервал 95% для групи gate_40: [0.178, 0.186]



1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?  
0.002 < 0.05

-> ймовірність, що різниця між gate_30 та gate_40 виникла випадково = 0.2%, що значно < 5%,  і тому ми відхиляємо H0.

  0.190201  >  0.182000

Ми впевнились, що стара версія дає краще утримання гравців.

Навіть якщо взяти найгірший варіант для gate_30 (0.187) і найкращий для gate_40 (0.186) — gate_30 все одно краща.

2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже? 

Довірчі інтервали різних версій гри не перетинаються. Різниця реальна навіть з урахуванням похибки. Тобто утримання гравців цих груп зовсім різне.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


In [18]:
import scipy.stats as stats

In [19]:
pd.crosstab(df["version"], df["retention_7"])

retention_7,False,True
version,,
gate_30,36198,8502
gate_40,37210,8279


In [20]:
crosstab = pd.crosstab(df["version"], df["retention_7"])

In [21]:
chi2, p, dof, expected = stats.chi2_contingency(crosstab)

print(f"χ² = {chi2:.3f}")
print(f"p-value = {p:.5f}")
print(f"Ступені свободи = {dof}")
print("Очікувані частоти:\n", expected)

χ² = 9.959
p-value = 0.00160
Ступені свободи = 1
Очікувані частоти:
 [[36382.90257127  8317.09742873]
 [37025.09742873  8463.90257127]]


H0​: між версією гри і retention_7 нема залежності

Ha​: між версією гри і retention_7 є залежність

p-value = 0.00160

α = 0.05 

0.00160 < 0.05

Маємо 0.16% що значно < за 5%,  і тому ми відхиляємо H0. Тобто, між версією гри і retention_7 дійсно є залежність.

Обидва тести дали однаковий висновок — стара версія з воротами на 30 рівні краще утримує гравців.
